# 01 · 재료 — KDS 14 20 재료 특성

KDS 14 20 이 규정하는 콘크리트·철근의 재료 상수를 확인하고, 응력-변형률
관계를 그린다.

| 항목 | 식 | 조문 |
|---|---|---|
| 콘크리트 탄성계수 | $E_c = 8500\sqrt[3]{f_{cm}}$ | KDS 14 20 10 4.3.3, 식 (4.3-2) |
| 평균압축강도 | $f_{cm} = f_{ck} + \Delta f$ | KDS 14 20 10 식 (4.3-3) |
| 등가직사각형 응력블록 | $\eta(0.85f_{ck})$, $a = \beta_1 c$ | KDS 14 20 20 4.1.1(8), 표 4.1-2 |
| 파괴계수 | $f_r = 0.63\lambda\sqrt{f_{ck}}$ | KDS 14 20 30 4.2.1 |
| 철근 탄성계수 | $E_s = 200{,}000$ MPa | KDS 14 20 10 4.3.3(2), 식 (4.3-5) |

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

## 재료 상수 표

`stress_block_parameters` 는 KDS 14 20 20 표 4.1-2 의 값을 반환한다.
표에 없는 강도는 선형보간한다.

In [ ]:
from concreteproperties_kds import (
    elastic_modulus,
    modulus_of_rupture,
    stress_block_parameters,
)

print(f"{'fck':>6} {'Ec':>10} {'eps_cu':>9} {'eta':>7} {'beta_1':>8}"
      f" {'0.85*eta*fck':>13} {'fr':>7}")
print("-" * 66)

for fck in [18, 21, 24, 27, 30, 35, 40, 50, 60, 70, 80, 90]:
    eps_cu, eta, beta_1 = stress_block_parameters(fck=fck)
    print(
        f"{fck:6.0f} {elastic_modulus(fck=fck):10.0f} {eps_cu:9.4f}"
        f" {eta:7.2f} {beta_1:8.2f} {0.85 * eta * fck:13.2f}"
        f" {modulus_of_rupture(fck=fck):7.2f}"
    )

고강도로 갈수록 $\varepsilon_{cu}$ 와 $\eta$, $\beta_1$ 이 모두 줄어든다.
콘크리트가 취성적이 되는 것을 반영한 것이다.

In [ ]:
fck = np.linspace(18, 90, 200)
eps_cu = [stress_block_parameters(f)[0] for f in fck]
eta = [stress_block_parameters(f)[1] for f in fck]
beta_1 = [stress_block_parameters(f)[2] for f in fck]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, y, name in zip(
    axes, [eps_cu, eta, beta_1], ["eps_cu", "eta", "beta_1"], strict=True
):
    ax.plot(fck, y)
    ax.set_xlabel("fck (MPa)")
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
fig.suptitle("KDS 14 20 20 Table 4.1-2")
fig.tight_layout()
plt.show()

## 변형률한계 (KDS 14 20 20 4.1.2)

- 압축지배변형률한계 $\varepsilon_y = f_y / E_s$ — 4.1.2(3)
- 인장지배변형률한계 0.005 ($f_y \le 400$) 또는 $2.5\varepsilon_y$ — 4.1.2(4)
- 휨부재 최소허용변형률 0.004 ($f_y \le 400$) 또는 $2.0\varepsilon_y$ — 4.1.2(5)

In [ ]:
from concreteproperties_kds import (
    compression_controlled_strain_limit,
    minimum_net_tensile_strain,
    tension_controlled_strain_limit,
)

print(f"{'강종':>8} {'fy':>6} {'eps_y':>9} {'eps_t,tl':>10} {'eps_t,min':>11}")
print("-" * 48)
for fy in [300, 400, 500, 600]:
    print(
        f"{'SD' + str(fy):>8} {fy:6.0f}"
        f" {compression_controlled_strain_limit(fy=fy):9.4f}"
        f" {tension_controlled_strain_limit(fy=fy):10.5f}"
        f" {minimum_net_tensile_strain(fy=fy):11.5f}"
    )

## 재료 객체와 응력-변형률 관계

`KDS.create_concrete_material` 은 사용(service)·극한(ultimate) 두 관계를
모두 갖춘 콘크리트 객체를 만든다.

In [ ]:
from concreteproperties_kds import KDS

kds = KDS()
conc = kds.create_concrete_material(compressive_strength=27)
steel = kds.create_steel_material(yield_strength=400)

print(conc.name)
print(f"  Ec = {conc.elastic_modulus:,.0f} MPa")
print(f"  fr = {conc.flexural_tensile_strength:.3f} MPa")
print(f"  단위질량 = {conc.density * 1e9:,.0f} kg/m^3")
print(steel.name)
print(f"  Es = {steel.elastic_modulus:,.0f} MPa")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))

conc.stress_strain_profile.plot_stress_strain(
    ax=axes[0], render=False, title="Concrete - service"
)
conc.ultimate_stress_strain_profile.plot_stress_strain(
    ax=axes[1], render=False, title="Concrete - ultimate"
)
steel.stress_strain_profile.plot_stress_strain(
    ax=axes[2], render=False, title="Steel SD400"
)
fig.tight_layout()
plt.show()

극한 관계의 압축응력이 $\eta(0.85 f_{ck}) = 1.0 \times 0.85 \times 27
= 22.95$ MPa 로 일정한 것을 볼 수 있다. 응력블록이 시작되는 변형률은
$\varepsilon_{cu}(1-\beta_1) = 0.0033 \times 0.2 = 0.00066$ 이다.